In [115]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Laplacian Interpolation and Laplacian Kernel Interpolation algorithms

In [116]:
def graph_laplacian(X):
    k=3 # n neighbors
    n_data = X.shape[0]
    W_matrix = np.zeros((n_data, n_data))
    
    # adjacency matrix of the graph (W)
    for i in range(n_data):
        # distance from that point to the rest of the graph
        distances = np.linalg.norm(X[i]-X, axis=1) # calculates euclidean distance
        distances[i] = np.inf # so it doesn't count itself as a neighbour
        neighbors_index = np.argsort(distances)[:k] # index of the nearest neighbors
        # update matrix
        for j in neighbors_index:
            W_matrix[i][j] = 1
            W_matrix[j][i] = 1
    
    # degree matrix of the graph (D)
    D_diag = np.sum(W_matrix, axis=1)
         
    return W_matrix, D_diag
    
    
def laplacian_interpolation(data, l):    
    # unlabelled indexes
    m = data.shape[0]
    u = np.setdiff1d(np.arange(m), l)
    
    # divide data into X and y (labels)
    X = data[:, 1:]
    y = data[:,0] - 2 # -2 so labels are -1 and 1
    
    # P=D^-1*W
    W_matrix, D_diag = graph_laplacian(X) # Adjacency matrix (W) and degree matrix (D) diagonal
    D_diag_inv = 1/D_diag  # D^-1
    P_matrix = D_diag_inv[:, np.newaxis]*W_matrix # P = D^-1*W
    # labels of labelled data
    f_l = y[l]
    # P_uu and P_ul
    P_uu = P_matrix[np.ix_(u, u)]
    P_ul = P_matrix[np.ix_(u, l)]
    
    # return labels of unsupervised data
    aux_matrix = np.eye(len(u)) - P_uu
    # solve because of numerical instability and computational reasons
    # np.linalg.inv(aux_matrix)@P_ul@f_l
    preds = np.linalg.solve(aux_matrix, P_ul@f_l) 
    # labels (sign of preds)
    labels = np.sign(preds)
    # compute error
    error = np.sum(labels != y[u])/(m-len(l))
    return error

def laplacian_kernel_interpolation(data, l):
    # unlabelled indexes
    m = data.shape[0]
    u = np.setdiff1d(np.arange(m), l)
    
    # divide data into X and y (labels)
    X = data[:, 1:]
    y = data[:,0] - 2
    
    # calculate graph laplacian pseudoinverse
    W_matrix, D_diag = graph_laplacian(X) # Adjacency matrix (W) and degree matrix (D) diagonal
    laplacian = np.diag(D_diag) - W_matrix
    L_plus = np.linalg.pinv(laplacian)
    # kernel matrix
    K_matrix = L_plus[np.ix_(l, l)]
    # labels of labelled data
    y_l = y[l]
    # alpha
    alpha = np.linalg.pinv(K_matrix)@y_l
    # labelled rows of L
    L_labelled = L_plus[l, :]
    # prediction vector
    v = alpha @ L_labelled
    # labels of unlabelled data
    v = np.sign(v[u])
    # compute error
    error = np.sum(v != y[u])/(m-len(l))
    return error

## Experiments

In [ ]:
datasets = ["dtrain13_50.dat", "dtrain13_100.dat", "dtrain13_200.dat", "dtrain13_400.dat"]
errors_li_dict = dict()
errors_lki_dict = dict()

# protocol
for filename in datasets:
    # load data
    dataset = np.loadtxt(filename)
    m = dataset.shape[0]
    # dictionary for the dataset
    errors_li_dict[filename] = dict()
    errors_lki_dict[filename] = dict()
    for l in [1, 2, 4, 8, 16]:
        errors_li = np.zeros(20)
        errors_lki = np.zeros(20)
        for i in range(20):
            # sample for first class
            sample_class1 = np.random.choice(np.arange(m/2), size=l, replace=False)
            # sample for second class
            sample_class2 = np.random.choice(np.arange(m/2), size=l, replace=False) + m/2
            # create (sorted) index list
            L = np.append(sample_class1, sample_class2).astype(int)
            L.sort()
            # errors for LI and LKI
            errLI = laplacian_interpolation(dataset, L)
            errLKI = laplacian_kernel_interpolation(dataset, L)
            # add to error list
            errors_li[i] = errLI
            errors_lki[i] = errLKI
        # error dictionary
        errors_li_dict[filename][l] = (round(float(np.mean(errors_li)), 3), round(float(np.std(errors_li)),3))
        errors_lki_dict[filename][l] = (round(float(np.mean(errors_lki)), 3), round(float(np.std(errors_lki)),3))
        

In [125]:
import json

with open("errors_li.json", "w") as f:
   json.dump(errors_li_dict, f) 
   
with open("errors_lki.json", "w") as f:
   json.dump(errors_lki_dict, f) 

In [122]:
errors_li_dict

{'dtrain13_50.dat': {1: (0.162, 0.105),
  2: (0.129, 0.092),
  4: (0.078, 0.048),
  8: (0.055, 0.029),
  16: (0.039, 0.019)},
 'dtrain13_100.dat': {1: (0.057, 0.034),
  2: (0.045, 0.021),
  4: (0.042, 0.014),
  8: (0.041, 0.015),
  16: (0.036, 0.015)},
 'dtrain13_200.dat': {1: (0.083, 0.095),
  2: (0.027, 0.016),
  4: (0.019, 0.007),
  8: (0.023, 0.008),
  16: (0.019, 0.007)},
 'dtrain13_400.dat': {1: (0.133, 0.188),
  2: (0.021, 0.014),
  4: (0.013, 0.002),
  8: (0.012, 0.002),
  16: (0.01, 0.002)}}

In [123]:
errors_lki_dict

{'dtrain13_50.dat': {1: (0.104, 0.061),
  2: (0.067, 0.032),
  4: (0.061, 0.035),
  8: (0.04, 0.009),
  16: (0.037, 0.016)},
 'dtrain13_100.dat': {1: (0.046, 0.019),
  2: (0.043, 0.02),
  4: (0.04, 0.013),
  8: (0.04, 0.014),
  16: (0.032, 0.009)},
 'dtrain13_200.dat': {1: (0.027, 0.027),
  2: (0.02, 0.005),
  4: (0.017, 0.005),
  8: (0.02, 0.005),
  16: (0.016, 0.005)},
 'dtrain13_400.dat': {1: (0.018, 0.027),
  2: (0.013, 0.004),
  4: (0.012, 0.002),
  8: (0.011, 0.001),
  16: (0.01, 0.002)}}